In [ ]:
import gc
import os
import time

import numpy as np
import optuna
import pandas as pd
import torch
import tsai.all as ts
from fastai.torch_core import set_seed
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Hyperparameter tuning -- HydraMultiROCKET, LSTM, TSSequencer

Nested-CV hyperparameter tuning for the three raw-accelerometer deep
learning models. The other models are tuned with the same validation
protocol, objective and trial budget in `07a_tuning_features_ml.ipynb`
(RF, XGBoost, SVC), `07b_tuning_features_dl.ipynb` (GANDALF,
CategoryEmbedding) and `07c_tuning_features_tabpfn.ipynb` (TabPFN). Each
model's cells can be run independently; outputs are merged into the
shared CSVs (see "Save outputs").

## Validation protocol

Uses the 5 stratified-group CV folds in `indices_cv_fold_lookup.csv`.
For each outer test fold, `N_INNER_FOLDS = 2` of the remaining 4 folds
(the two folds following the test fold, cyclically) each serve once as
the inner validation fold, training on the other 3 non-test folds. Each
Optuna trial is scored by the mean validation f1_macro over these 2
inner folds. The best configuration is then retrained on all 4 non-test
folds and evaluated once on the test fold, which is never used during
the search.

## Search spaces

- **HydraMultiROCKET**: `fc_dropout` (float, 0.0-0.5) and `num_features`
  (integer, 10000-50000, log scale; the upper bound is the tsai
  default). `g` is not passed, so tsai's default applies.
- **LSTM**: `hidden_size` (integer, 50-300) and `fc_dropout` (float,
  0.0-0.5). `n_layers=6` and `bidirectional=True` are fixed, as in
  `05d_timeseries_classification_dl.ipynb`.
- **TSSequencer**: `d_model` (integer, 32-256) and `fc_dropout` (float,
  0.0-0.5). `depth` is not passed, so tsai's default applies.

All models, as in `05d`: learning rate chosen per training run with
`lr_find()`, 50 epochs of `fit_one_cycle`, batch sizes [64, 128],
class-weighted cross-entropy loss.

## Objective, budget and seeding

Objective: mean inner-validation f1_macro, maximised with Optuna's TPE
sampler (`multivariate=True`, `seed=RANDOM_STATE`). One study per outer
fold and model, with `N_TRIALS = 50` trials, i.e. 50 x 2 inner trainings
plus 2 final retrains (tuned and default) per outer fold.

Before every inner-fold training and every final retrain,
`set_seed(RANDOM_STATE, reproducible=True)` is called, as in `05d`. This
fastai helper seeds Python's `random`, NumPy (which tsai's dataloaders
use for batch shuffling) and torch (CPU and CUDA), and makes cuDNN
deterministic, so all trials and both final retrains start from the same
random state.

## Default-hyperparameter controls

Each model's `*_DEFAULT_PARAMS` (the configuration used in `05d`) is
enqueued as the first of the `N_TRIALS` trials, so its inner-validation
score (`default_val_f1_macro`) can be compared with the winner's
(`val_f1_macro`).

The defaults are also retrained on the same 4 non-test folds and
evaluated on the same test fold, through the same code path and seed as
the tuned winner (`retrain_and_evaluate_*`), so tuned vs default is
like-for-like. These rows (`model="<name>_default"`) are saved to
separate `..._tuned_default_control_*.csv` files, so the
`..._tuned_metrics.csv` / `..._tuned_confusion.csv` files read by
`12b_publication_plots_fig3.R` contain only tuned rows.

Each outer fold's Optuna study is kept in memory for the "Inspect best
study per dataset" cells (summary and plots for the outer fold with the
best validation f1_macro).

## Scope

Burst_1/uncorrected only (`DATASETS`), behaviour target only.

## Load the main CV fold lookup and set tuning constants

In [ ]:
FOLD_LOOKUP_PATH = "../data/temp/indices_cv_fold_lookup.csv"
DATASETS = [
    ("burst_1", "uncorrected"),
]
# Optuna trials per outer fold and model (includes the enqueued default).
N_TRIALS = 50
RANDOM_STATE = 42
# Number of inner validation folds whose f1_macro is averaged per Optuna
# trial (folds (i+1) and (i+2) mod 5 for outer test fold i).
N_INNER_FOLDS = 2
# Default-hyperparameter controls. Each is enqueued as one of the N_TRIALS
# trials and also retrained/evaluated on every test fold.
# HydraMultiROCKET: tsai defaults, as used (no kwargs) in
# 05d_timeseries_classification_dl.ipynb. g is never passed.
HR_DEFAULT_PARAMS = {"fc_dropout": 0.0, "num_features": 50000}
# LSTM: tsai defaults for hidden_size/fc_dropout. n_layers=6 and
# bidirectional=True (as in 05d) are fixed in every ts.create_model call.
LSTM_DEFAULT_PARAMS = {"hidden_size": 100, "fc_dropout": 0.0}
# TSSequencer: tsai defaults for d_model/fc_dropout, as used (no kwargs) in
# 05d. depth is never passed.
TSSEQUENCER_DEFAULT_PARAMS = {"d_model": 128, "fc_dropout": 0.0}

fold_lookup_all = pd.read_csv(FOLD_LOOKUP_PATH)
print(
    f"Loaded fold lookup: {len(fold_lookup_all)} rows, "
    f"bursts={sorted(fold_lookup_all['burst'].unique())}"
)

## Build the time-series X array once, and a flexible train/valid split helper

In [ ]:
def build_ts_arrays(acc, fold_lookup, burst):
    """Build the time-series X array once per file, aligned to fold assignments.

    Matches build_ts_arrays in 05d_timeseries_classification_dl.ipynb.
    fold_lookup (indices_cv_fold_lookup.csv) contains all bursts, so it is
    filtered to `burst` here.

    Args:
        acc (pd.DataFrame): Raw accelerometer long-format data straight from
            the parquet file (columns: Ind_ID, new_burst, feature, plus
            numeric feature/time columns -- no "id" column of its own).
        fold_lookup (pd.DataFrame): Fold lookup table with columns
            burst, Ind_ID, new_burst, behaviour, fold
        burst (str): Folder-derived burst name (e.g. "Burst_1"); matched
            against fold_lookup's lowercase burst column

    Returns:
        tuple: (X, merged) where X is an (n_samples, 3, n_features) array and
            merged is a DataFrame with columns id, Ind_ID, new_burst, idx,
            fold, behaviour -- merged.loc[i, "idx"] indexes X's
            first axis for sample i.
    """
    m = fold_lookup.loc[
        fold_lookup["burst"] == burst.lower(),
        ["Ind_ID", "new_burst", "behaviour", "fold"],
    ]

    acc = acc.copy()
    acc["new_burst"] = acc["new_burst"].astype(np.int64)
    acc["id"] = acc["Ind_ID"].astype(str) + "_" + acc["new_burst"].astype(str)
    id_mapping = (
        acc[["id", "Ind_ID", "new_burst"]].drop_duplicates().reset_index(drop=True)
    )

    merged = id_mapping.merge(m, on=["Ind_ID", "new_burst"], how="inner")
    merged = merged.reset_index(drop=True)
    merged["idx"] = merged.index

    feature_cols = [col for col in acc.columns if col.isdigit()]
    n_features = len(feature_cols)
    n_samples = len(merged)
    print(f"Number of samples: {n_samples}, number of features: {n_features}")

    X = np.zeros((n_samples, 3, n_features), dtype=np.float64)
    feature_to_idx = {"X": 0, "Y": 1, "Z": 2}

    for i, id_val in enumerate(merged["id"]):
        id_data = acc[acc["id"] == id_val]
        for feature_type, axis_idx in feature_to_idx.items():
            feature_data = id_data[id_data["feature"] == feature_type]
            if len(feature_data) > 0:
                X[i, axis_idx, :] = feature_data[feature_cols].values[0]
            else:
                print(f"Warning: No data found for ID {id_val}, feature {feature_type}")

    return X.astype(np.float64), merged


def create_dsets_for_split(X, merged, target_col, train_folds, valid_folds):
    """Create a TSDatasets object from arbitrary train/valid fold-id groups.

    Generalizes create_dsets_for_fold (05d_timeseries_classification_dl.ipynb)
    from "all folds except f" / "f" to arbitrary fold-id lists, so the same
    helper builds both the search split (3 train folds -> 1 val fold) and
    the final split (4 folds -> 1 test fold).

    Args:
        X (np.ndarray): Prebuilt (n_samples, 3, n_features) array shared
            across folds (see build_ts_arrays)
        merged (pd.DataFrame): Output of build_ts_arrays, with idx/fold/
            behaviour columns
        target_col (str): Label column ("behaviour" in this notebook)
        train_folds (list): Fold ids to use as the training split
        valid_folds (list): Fold ids to use as the validation/test split

    Returns:
        TSDatasets: Dataset object ready for use with tsai DataLoaders
    """
    train_indices = merged.loc[merged["fold"].isin(train_folds), "idx"].tolist()
    valid_indices = merged.loc[merged["fold"].isin(valid_folds), "idx"].tolist()
    splits = (train_indices, valid_indices)

    y = merged[target_col].to_numpy(dtype="<U20")

    tfms = [None, [ts.Categorize()]]
    dsets = ts.TSDatasets(
        X.astype(np.float64), y=y, tfms=tfms, splits=splits, inplace=True
    )
    return dsets

## HydraMultiROCKET Optuna search + final evaluation, for one outer test fold

In [ ]:
def make_hr_objective(
    X, merged, col, non_test_folds, val_folds_list, use_class_weight=True
):
    """Build an Optuna objective returning the mean f1_macro across the
    N_INNER_FOLDS validation folds in val_folds_list.

    Searches fc_dropout and num_features. g is not passed to
    ts.create_model, so tsai's default applies.

    Args:
        non_test_folds (list): all fold ids except the current test fold.
        val_folds_list (list): N_INNER_FOLDS fold ids (subset of
            non_test_folds), each used once as the inner validation fold,
            training on the remaining non_test_folds.
    """

    def objective(trial):
        fc_dropout = trial.suggest_float("fc_dropout", 0.0, 0.5)
        # Capped at the tsai default (50000), so no candidate costs more
        # than the default configuration.
        num_features = trial.suggest_int("num_features", 10000, 50000, log=True)

        scores = []
        for val_fold in val_folds_list:
            train_folds = [f for f in non_test_folds if f != val_fold]

            # Re-seed before each inner fold's data/model construction, so
            # every trial and inner fold starts from the same random state
            # (including HydraMultiROCKET's random kernel bank, built in
            # ts.create_model, and the batch order).
            set_seed(RANDOM_STATE, reproducible=True)

            dset = create_dsets_for_split(X, merged, col, train_folds, [val_fold])
            dls = ts.TSDataLoaders.from_dsets(dset.train, dset.valid, bs=[64, 128])
            loss_func = (
                ts.CrossEntropyLossFlat(weight=dls.train.cws)
                if use_class_weight
                else None
            )

            model = ts.create_model(
                ts.HydraMultiRocket,
                dls=dls,
                fc_dropout=fc_dropout,
                num_features=num_features,
            )
            learn = ts.Learner(dls, model, metrics=[], loss_func=loss_func)
            lr_max = learn.lr_find()
            learn.fit_one_cycle(50, lr_max)

            _, y_true, y_pred = learn.get_preds(with_decoded=True)
            y_true = y_true.cpu().numpy()
            y_pred = y_pred.cpu().numpy()
            scores.append(f1_score(y_true, y_pred, average="macro", zero_division=0))

        return float(np.mean(scores))

    return objective


def retrain_and_evaluate_hr(dls, params, use_class_weight=True):
    """Fit one HydraMultiROCKET config on dls.train and evaluate once on
    dls.valid, returning (metrics_row, y_true, y_pred).

    Used for both final retrains in tune_and_evaluate_hr_fold (tuned
    winner and HR_DEFAULT_PARAMS), so both use the same code path, split
    and seed and differ only in hyperparameters. Re-seeds with
    set_seed(RANDOM_STATE, reproducible=True) before model construction.
    """
    set_seed(RANDOM_STATE, reproducible=True)

    metrics = [
        ts.accuracy,
        ts.BalancedAccuracy(),
        ts.F1Score(average="weighted"),
        ts.Precision(average="weighted"),
        ts.Recall(average="weighted"),
        ts.F1Score(average="macro"),
        ts.Precision(average="macro"),
        ts.Recall(average="macro"),
    ]
    loss_func = (
        ts.CrossEntropyLossFlat(weight=dls.train.cws) if use_class_weight else None
    )

    model = ts.create_model(ts.HydraMultiRocket, dls=dls, **params)
    learn = ts.Learner(dls, model, metrics=metrics, loss_func=loss_func)

    t0 = time.time()
    lr_max = learn.lr_find()
    learn.fit_one_cycle(50, lr_max)
    elapsed = time.time() - t0

    vals = learn.recorder.values[-1]

    y_proba, y_true, y_pred = learn.get_preds(with_decoded=True)
    y_proba = y_proba.cpu().numpy()
    y_true = y_true.cpu().numpy()
    y_pred = y_pred.cpu().numpy()

    try:
        roc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted")
    except Exception:
        roc = np.nan

    metrics_row = {
        "fc_dropout": params["fc_dropout"],
        "num_features": params["num_features"],
        "train_loss": vals[0],
        "valid_loss": vals[1],
        "accuracy": vals[2],
        "roc_auc": roc,
        "balanced_acc": vals[3],
        "f1_score": vals[4],
        "precision": vals[5],
        "recall": vals[6],
        "f1_macro": vals[7],
        "precision_macro": vals[8],
        "recall_macro": vals[9],
        "time": elapsed,
    }
    return metrics_row, y_true, y_pred


def tune_and_evaluate_hr_fold(
    X,
    merged,
    col,
    non_test_folds,
    val_folds_list,
    test_fold,
    burst,
    correction_type,
    n_trials=N_TRIALS,
    use_class_weight=True,
):
    """Run one outer test fold for HydraMultiROCKET.

    Optuna searches hyperparameters scored by the mean f1_macro across
    N_INNER_FOLDS validation folds (see make_hr_objective). HR_DEFAULT_PARAMS
    is enqueued as one of the n_trials trials, giving its validation score
    (default_val_f1_macro). The winner is retrained on all non_test_folds
    and evaluated once on test_fold; HR_DEFAULT_PARAMS is retrained and
    evaluated on the same split with the same seed (see
    retrain_and_evaluate_hr). Predictions come from learn.get_preds; the
    confusion matrix and ROC AUC are computed with sklearn.

    One Optuna study per outer fold (TPE, multivariate=True), named by
    burst/correction_type/test_fold. The study is returned for post-hoc
    inspection (see the "Inspect best study per dataset" cells).

    Returns:
        tuple: (metrics_df, confusion_df, default_metrics_df,
            default_confusion_df, study). metrics_df/confusion_df cover
            the tuned winner (model="HydraMultiRocket");
            default_metrics_df/default_confusion_df cover the
            HR_DEFAULT_PARAMS retrain (model="HydraMultiRocket_default"),
            kept separate and saved to the ..._tuned_default_control_*
            files.
    """
    study_name = f"{burst.lower()}_{correction_type}_testfold{test_fold}"
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
    study = optuna.create_study(
        direction="maximize", sampler=sampler, study_name=study_name
    )

    study.enqueue_trial(HR_DEFAULT_PARAMS)

    search_start = time.time()
    study.optimize(
        make_hr_objective(
            X,
            merged,
            col,
            non_test_folds,
            val_folds_list,
            use_class_weight=use_class_weight,
        ),
        n_trials=n_trials,
    )
    search_time = time.time() - search_start
    best_params = study.best_params

    # Validation score of the enqueued default trial (matched on params).
    default_trial = next(
        (t for t in study.trials if t.params == HR_DEFAULT_PARAMS), None
    )
    default_val_f1_macro = default_trial.value if default_trial is not None else np.nan

    # Parameter importances for this outer fold; wrapped in try/except so a
    # failure (e.g. degenerate trials) doesn't stop the run.
    try:
        importances = optuna.importance.get_param_importances(study)
    except Exception as e:
        print(f"  Could not compute parameter importances: {e!s}")
        importances = {}

    results = []
    confusion_data = []
    default_results = []
    default_confusion_data = []
    model_name = "HydraMultiRocket"

    try:
        dset = create_dsets_for_split(X, merged, col, non_test_folds, [test_fold])
        dls = ts.TSDataLoaders.from_dsets(dset.train, dset.valid, bs=[64, 128])
        classes = dset.vocab
        n_classes = len(classes)

        # retrain_and_evaluate_hr re-seeds before each call.
        tuned_row, y_true, y_pred = retrain_and_evaluate_hr(
            dls, best_params, use_class_weight=use_class_weight
        )
        tuned_row.update(
            {
                "model": model_name,
                "val_f1_macro": study.best_value,
                "default_val_f1_macro": default_val_f1_macro,
                "importance_fc_dropout": importances.get("fc_dropout", np.nan),
                "importance_num_features": importances.get("num_features", np.nan),
                "search_time": search_time,
            }
        )
        results.append(tuned_row)

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
        for actual_idx, actual in enumerate(classes):
            for pred_idx, predicted in enumerate(classes):
                confusion_data.append(
                    {
                        "model": model_name,
                        "actual_label": actual,
                        "predicted_label": predicted,
                        "count": cm[actual_idx, pred_idx],
                    }
                )

        # Default control, retrained on the same dls as the tuned winner.
        default_row, default_y_true, default_y_pred = retrain_and_evaluate_hr(
            dls, HR_DEFAULT_PARAMS, use_class_weight=use_class_weight
        )
        default_row["model"] = f"{model_name}_default"
        default_results.append(default_row)

        default_cm = confusion_matrix(
            default_y_true, default_y_pred, labels=np.arange(n_classes)
        )
        for actual_idx, actual in enumerate(classes):
            for pred_idx, predicted in enumerate(classes):
                default_confusion_data.append(
                    {
                        "model": f"{model_name}_default",
                        "actual_label": actual,
                        "predicted_label": predicted,
                        "count": default_cm[actual_idx, pred_idx],
                    }
                )
    except Exception as e:
        print(f"Error with {model_name}: {e!s}")

    results_df = pd.DataFrame(results)
    confusion_df = pd.DataFrame(confusion_data)
    default_results_df = pd.DataFrame(default_results)
    default_confusion_df = pd.DataFrame(default_confusion_data)

    return results_df, confusion_df, default_results_df, default_confusion_df, study

## HydraMultiROCKET tuning orchestration, across the 5 outer folds

In [ ]:
def process_parquet_file_hr(
    file_path, fold_lookup_all, n_trials=N_TRIALS, use_class_weight=True
):
    """Run the nested-CV protocol (see notebook intro) for
    HydraMultiROCKET hyperparameter tuning + final evaluation, for the
    behaviour target on one file.

    Same protocol as 07a/07b/07c's process_parquet_file_* functions, using the
    raw accelerometer time-series pipeline (build_ts_arrays / TSDatasets).
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    acc = pd.read_parquet(file_path)
    X, merged = build_ts_arrays(acc, fold_lookup_all, burst)
    folds = sorted(merged["fold"].unique())
    n_folds = len(folds)

    out = {
        "multiclass_metrics": [],
        "multiclass_conf": [],
        "default_metrics": [],
        "default_conf": [],
        "studies": {},
    }
    col, tgt = "behaviour", "Behaviour"
    for i, test_fold in enumerate(folds):
        non_test_folds = [f for f in folds if f != test_fold]
        val_folds_list = [folds[(i + k) % n_folds] for k in range(1, N_INNER_FOLDS + 1)]
        n_test = int((merged["fold"] == test_fold).sum())

        print(
            f"  Tuning {tgt} classification, test_fold={test_fold}, "
            f"inner val folds={val_folds_list}, non_test_folds={non_test_folds}"
        )
        met, conf, default_met, default_conf, study = tune_and_evaluate_hr_fold(
            X,
            merged,
            col,
            non_test_folds,
            val_folds_list,
            test_fold,
            burst,
            correction_type,
            n_trials=n_trials,
            use_class_weight=use_class_weight,
        )
        out["studies"][test_fold] = study
        if met.empty:
            # Final retrain failed (error already printed): skip, since a
            # columnless frame would break the concat below. The study is
            # still kept.
            continue
        for d in (met, conf, default_met, default_conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = tgt
            d["test_fold"] = test_fold
            d["val_folds"] = ",".join(str(vf) for vf in val_folds_list)
            d["n_test_samples"] = n_test
        out["multiclass_metrics"].append(met)
        out["multiclass_conf"].append(conf)
        out["default_metrics"].append(default_met)
        out["default_conf"].append(default_conf)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "fc_dropout",
        "num_features",
        "val_f1_macro",
        "default_val_f1_macro",
        "importance_fc_dropout",
        "importance_num_features",
        "train_loss",
        "valid_loss",
        "accuracy",
        "roc_auc",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "search_time",
        "time",
    ]
    default_metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "fc_dropout",
        "num_features",
        "train_loss",
        "valid_loss",
        "accuracy",
        "roc_auc",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: (
        pd.concat([d for d in lst if not d.empty])[cols]
        if any(not d.empty for d in lst)
        else pd.DataFrame(columns=cols)
    )
    return {
        "multiclass_metrics": fin(out["multiclass_metrics"], metric_columns),
        "multiclass_conf": fin(out["multiclass_conf"], conf_columns),
        "default_metrics": fin(out["default_metrics"], default_metric_columns),
        "default_conf": fin(out["default_conf"], conf_columns),
        "studies": out["studies"],
    }

## Run tuning across datasets

In [ ]:
all_results_hr = []
for burst, correction_type in DATASETS:
    pq_file_path = (
        f"../data/temp/acc_tsai/{burst.capitalize()}/"
        f"annotated_acc_tsai_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file_hr(pq_file_path, fold_lookup_all, n_trials=N_TRIALS)
    all_results_hr.append(result)

multiclass_metrics_hr = pd.concat(
    [r["multiclass_metrics"] for r in all_results_hr], ignore_index=True
)
multiclass_conf_hr = pd.concat(
    [r["multiclass_conf"] for r in all_results_hr], ignore_index=True
)

# Default control (model="HydraMultiRocket_default"), retrained on the same
# splits as the tuned winner. Kept separate from multiclass_metrics_hr/
# multiclass_conf_hr (saved to separate files).
default_metrics_hr = pd.concat(
    [r["default_metrics"] for r in all_results_hr], ignore_index=True
)
default_conf_hr = pd.concat(
    [r["default_conf"] for r in all_results_hr], ignore_index=True
)

# Optuna studies kept in memory (not saved) for the "Inspect best study
# per dataset (HydraMultiROCKET)" cell.
all_studies_hr = {
    dataset_key: result["studies"]
    for dataset_key, result in zip(DATASETS, all_results_hr)
}

## LSTM Optuna search + final evaluation, for one outer test fold

In [ ]:
def make_lstm_objective(
    X, merged, col, non_test_folds, val_folds_list, use_class_weight=True
):
    """Build an Optuna objective returning the mean f1_macro across the
    N_INNER_FOLDS validation folds in val_folds_list. Same as
    make_hr_objective except for the architecture (ts.LSTM) and search
    space (hidden_size, fc_dropout).

    n_layers=6/bidirectional=True are fixed, as in
    05d_timeseries_classification_dl.ipynb.

    Args:
        non_test_folds (list): all fold ids except the current test fold.
        val_folds_list (list): N_INNER_FOLDS fold ids (subset of
            non_test_folds), each used once as the inner validation fold,
            training on the remaining non_test_folds.
    """

    def objective(trial):
        hidden_size = trial.suggest_int("hidden_size", 50, 300)
        fc_dropout = trial.suggest_float("fc_dropout", 0.0, 0.5)

        scores = []
        for val_fold in val_folds_list:
            train_folds = [f for f in non_test_folds if f != val_fold]

            # Re-seed before each inner fold, as in make_hr_objective.
            set_seed(RANDOM_STATE, reproducible=True)

            dset = create_dsets_for_split(X, merged, col, train_folds, [val_fold])
            dls = ts.TSDataLoaders.from_dsets(dset.train, dset.valid, bs=[64, 128])
            loss_func = (
                ts.CrossEntropyLossFlat(weight=dls.train.cws)
                if use_class_weight
                else None
            )

            model = ts.create_model(
                ts.LSTM,
                dls=dls,
                hidden_size=hidden_size,
                fc_dropout=fc_dropout,
                n_layers=6,
                bidirectional=True,
            )
            learn = ts.Learner(dls, model, metrics=[], loss_func=loss_func)
            lr_max = learn.lr_find()
            learn.fit_one_cycle(50, lr_max)

            _, y_true, y_pred = learn.get_preds(with_decoded=True)
            y_true = y_true.cpu().numpy()
            y_pred = y_pred.cpu().numpy()
            scores.append(f1_score(y_true, y_pred, average="macro", zero_division=0))

        return float(np.mean(scores))

    return objective


def retrain_and_evaluate_lstm(dls, params, use_class_weight=True):
    """Fit one LSTM config on dls.train and evaluate once on dls.valid,
    returning (metrics_row, y_true, y_pred). Same as
    retrain_and_evaluate_hr; used for both final retrains (tuned winner
    and LSTM_DEFAULT_PARAMS). n_layers=6/bidirectional=True are fixed.
    """
    set_seed(RANDOM_STATE, reproducible=True)

    metrics = [
        ts.accuracy,
        ts.BalancedAccuracy(),
        ts.F1Score(average="weighted"),
        ts.Precision(average="weighted"),
        ts.Recall(average="weighted"),
        ts.F1Score(average="macro"),
        ts.Precision(average="macro"),
        ts.Recall(average="macro"),
    ]
    loss_func = (
        ts.CrossEntropyLossFlat(weight=dls.train.cws) if use_class_weight else None
    )

    model = ts.create_model(
        ts.LSTM,
        dls=dls,
        hidden_size=params["hidden_size"],
        fc_dropout=params["fc_dropout"],
        n_layers=6,
        bidirectional=True,
    )
    learn = ts.Learner(dls, model, metrics=metrics, loss_func=loss_func)

    t0 = time.time()
    lr_max = learn.lr_find()
    learn.fit_one_cycle(50, lr_max)
    elapsed = time.time() - t0

    vals = learn.recorder.values[-1]

    y_proba, y_true, y_pred = learn.get_preds(with_decoded=True)
    y_proba = y_proba.cpu().numpy()
    y_true = y_true.cpu().numpy()
    y_pred = y_pred.cpu().numpy()

    try:
        roc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted")
    except Exception:
        roc = np.nan

    metrics_row = {
        "hidden_size": params["hidden_size"],
        "fc_dropout": params["fc_dropout"],
        "train_loss": vals[0],
        "valid_loss": vals[1],
        "accuracy": vals[2],
        "roc_auc": roc,
        "balanced_acc": vals[3],
        "f1_score": vals[4],
        "precision": vals[5],
        "recall": vals[6],
        "f1_macro": vals[7],
        "precision_macro": vals[8],
        "recall_macro": vals[9],
        "time": elapsed,
    }
    return metrics_row, y_true, y_pred


def tune_and_evaluate_lstm_fold(
    X,
    merged,
    col,
    non_test_folds,
    val_folds_list,
    test_fold,
    burst,
    correction_type,
    n_trials=N_TRIALS,
    use_class_weight=True,
):
    """Run one outer test fold for LSTM. Same protocol as
    tune_and_evaluate_hr_fold (see its docstring), using
    make_lstm_objective, retrain_and_evaluate_lstm and LSTM_DEFAULT_PARAMS.

    Returns:
        tuple: (metrics_df, confusion_df, default_metrics_df,
            default_confusion_df, study). Same shape as
            tune_and_evaluate_hr_fold's return, with model="LSTM" /
            "LSTM_default".
    """
    study_name = f"{burst.lower()}_{correction_type}_testfold{test_fold}"
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
    study = optuna.create_study(
        direction="maximize", sampler=sampler, study_name=study_name
    )

    study.enqueue_trial(LSTM_DEFAULT_PARAMS)

    search_start = time.time()
    study.optimize(
        make_lstm_objective(
            X,
            merged,
            col,
            non_test_folds,
            val_folds_list,
            use_class_weight=use_class_weight,
        ),
        n_trials=n_trials,
    )
    search_time = time.time() - search_start
    best_params = study.best_params

    default_trial = next(
        (t for t in study.trials if t.params == LSTM_DEFAULT_PARAMS), None
    )
    default_val_f1_macro = default_trial.value if default_trial is not None else np.nan

    try:
        importances = optuna.importance.get_param_importances(study)
    except Exception as e:
        print(f"  Could not compute parameter importances: {e!s}")
        importances = {}

    results = []
    confusion_data = []
    default_results = []
    default_confusion_data = []
    model_name = "LSTM"

    try:
        dset = create_dsets_for_split(X, merged, col, non_test_folds, [test_fold])
        dls = ts.TSDataLoaders.from_dsets(dset.train, dset.valid, bs=[64, 128])
        classes = dset.vocab
        n_classes = len(classes)

        tuned_row, y_true, y_pred = retrain_and_evaluate_lstm(
            dls, best_params, use_class_weight=use_class_weight
        )
        tuned_row.update(
            {
                "model": model_name,
                "val_f1_macro": study.best_value,
                "default_val_f1_macro": default_val_f1_macro,
                "importance_hidden_size": importances.get("hidden_size", np.nan),
                "importance_fc_dropout": importances.get("fc_dropout", np.nan),
                "search_time": search_time,
            }
        )
        results.append(tuned_row)

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
        for actual_idx, actual in enumerate(classes):
            for pred_idx, predicted in enumerate(classes):
                confusion_data.append(
                    {
                        "model": model_name,
                        "actual_label": actual,
                        "predicted_label": predicted,
                        "count": cm[actual_idx, pred_idx],
                    }
                )

        default_row, default_y_true, default_y_pred = retrain_and_evaluate_lstm(
            dls, LSTM_DEFAULT_PARAMS, use_class_weight=use_class_weight
        )
        default_row["model"] = f"{model_name}_default"
        default_results.append(default_row)

        default_cm = confusion_matrix(
            default_y_true, default_y_pred, labels=np.arange(n_classes)
        )
        for actual_idx, actual in enumerate(classes):
            for pred_idx, predicted in enumerate(classes):
                default_confusion_data.append(
                    {
                        "model": f"{model_name}_default",
                        "actual_label": actual,
                        "predicted_label": predicted,
                        "count": default_cm[actual_idx, pred_idx],
                    }
                )
    except Exception as e:
        print(f"Error with {model_name}: {e!s}")

    results_df = pd.DataFrame(results)
    confusion_df = pd.DataFrame(confusion_data)
    default_results_df = pd.DataFrame(default_results)
    default_confusion_df = pd.DataFrame(default_confusion_data)

    return results_df, confusion_df, default_results_df, default_confusion_df, study

## LSTM tuning orchestration, across the 5 outer folds

In [ ]:
def process_parquet_file_lstm(
    file_path, fold_lookup_all, n_trials=N_TRIALS, use_class_weight=True
):
    """Run the nested-CV protocol (see notebook intro) for LSTM
    hyperparameter tuning + final evaluation, for the behaviour target on
    one file.

    Same as process_parquet_file_hr except for the per-fold trainer
    (tune_and_evaluate_lstm_fold) and hyperparameter columns (hidden_size,
    fc_dropout).
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    acc = pd.read_parquet(file_path)
    X, merged = build_ts_arrays(acc, fold_lookup_all, burst)
    folds = sorted(merged["fold"].unique())
    n_folds = len(folds)

    out = {
        "multiclass_metrics": [],
        "multiclass_conf": [],
        "default_metrics": [],
        "default_conf": [],
        "studies": {},
    }
    col, tgt = "behaviour", "Behaviour"
    for i, test_fold in enumerate(folds):
        non_test_folds = [f for f in folds if f != test_fold]
        val_folds_list = [folds[(i + k) % n_folds] for k in range(1, N_INNER_FOLDS + 1)]
        n_test = int((merged["fold"] == test_fold).sum())

        print(
            f"  Tuning {tgt} classification, test_fold={test_fold}, "
            f"inner val folds={val_folds_list}, non_test_folds={non_test_folds}"
        )
        met, conf, default_met, default_conf, study = tune_and_evaluate_lstm_fold(
            X,
            merged,
            col,
            non_test_folds,
            val_folds_list,
            test_fold,
            burst,
            correction_type,
            n_trials=n_trials,
            use_class_weight=use_class_weight,
        )
        out["studies"][test_fold] = study
        if met.empty:
            # Final retrain failed: skip (see process_parquet_file_hr).
            continue
        for d in (met, conf, default_met, default_conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = tgt
            d["test_fold"] = test_fold
            d["val_folds"] = ",".join(str(vf) for vf in val_folds_list)
            d["n_test_samples"] = n_test
        out["multiclass_metrics"].append(met)
        out["multiclass_conf"].append(conf)
        out["default_metrics"].append(default_met)
        out["default_conf"].append(default_conf)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "hidden_size",
        "fc_dropout",
        "val_f1_macro",
        "default_val_f1_macro",
        "importance_hidden_size",
        "importance_fc_dropout",
        "train_loss",
        "valid_loss",
        "accuracy",
        "roc_auc",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "search_time",
        "time",
    ]
    default_metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "hidden_size",
        "fc_dropout",
        "train_loss",
        "valid_loss",
        "accuracy",
        "roc_auc",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: (
        pd.concat([d for d in lst if not d.empty])[cols]
        if any(not d.empty for d in lst)
        else pd.DataFrame(columns=cols)
    )
    return {
        "multiclass_metrics": fin(out["multiclass_metrics"], metric_columns),
        "multiclass_conf": fin(out["multiclass_conf"], conf_columns),
        "default_metrics": fin(out["default_metrics"], default_metric_columns),
        "default_conf": fin(out["default_conf"], conf_columns),
        "studies": out["studies"],
    }

## Run LSTM tuning across datasets

In [ ]:
all_results_lstm = []
for burst, correction_type in DATASETS:
    pq_file_path = (
        f"../data/temp/acc_tsai/{burst.capitalize()}/"
        f"annotated_acc_tsai_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file_lstm(pq_file_path, fold_lookup_all, n_trials=N_TRIALS)
    all_results_lstm.append(result)

multiclass_metrics_lstm = pd.concat(
    [r["multiclass_metrics"] for r in all_results_lstm], ignore_index=True
)
multiclass_conf_lstm = pd.concat(
    [r["multiclass_conf"] for r in all_results_lstm], ignore_index=True
)
default_metrics_lstm = pd.concat(
    [r["default_metrics"] for r in all_results_lstm], ignore_index=True
)
default_conf_lstm = pd.concat(
    [r["default_conf"] for r in all_results_lstm], ignore_index=True
)

# Optuna studies kept in memory (not saved) for the "Inspect best study
# per dataset (LSTM)" cell.
all_studies_lstm = {
    dataset_key: result["studies"]
    for dataset_key, result in zip(DATASETS, all_results_lstm)
}

## TSSequencer Optuna search + final evaluation, for one outer test fold

In [ ]:
def make_tssequencer_objective(
    X, merged, col, non_test_folds, val_folds_list, use_class_weight=True
):
    """Build an Optuna objective returning the mean f1_macro across the
    N_INNER_FOLDS validation folds in val_folds_list. Same as
    make_hr_objective except for the architecture (ts.TSSequencer) and
    search space (d_model, fc_dropout).

    depth is not passed, so tsai's default applies.

    Args:
        non_test_folds (list): all fold ids except the current test fold.
        val_folds_list (list): N_INNER_FOLDS fold ids (subset of
            non_test_folds), each used once as the inner validation fold,
            training on the remaining non_test_folds.
    """

    def objective(trial):
        d_model = trial.suggest_int("d_model", 32, 256)
        fc_dropout = trial.suggest_float("fc_dropout", 0.0, 0.5)

        scores = []
        for val_fold in val_folds_list:
            train_folds = [f for f in non_test_folds if f != val_fold]

            # Re-seed before each inner fold, as in make_hr_objective.
            set_seed(RANDOM_STATE, reproducible=True)

            dset = create_dsets_for_split(X, merged, col, train_folds, [val_fold])
            dls = ts.TSDataLoaders.from_dsets(dset.train, dset.valid, bs=[64, 128])
            loss_func = (
                ts.CrossEntropyLossFlat(weight=dls.train.cws)
                if use_class_weight
                else None
            )

            model = ts.create_model(
                ts.TSSequencer,
                dls=dls,
                d_model=d_model,
                fc_dropout=fc_dropout,
            )
            learn = ts.Learner(dls, model, metrics=[], loss_func=loss_func)
            lr_max = learn.lr_find()
            learn.fit_one_cycle(50, lr_max)

            _, y_true, y_pred = learn.get_preds(with_decoded=True)
            y_true = y_true.cpu().numpy()
            y_pred = y_pred.cpu().numpy()
            scores.append(f1_score(y_true, y_pred, average="macro", zero_division=0))

        return float(np.mean(scores))

    return objective


def retrain_and_evaluate_tssequencer(dls, params, use_class_weight=True):
    """Fit one TSSequencer config on dls.train and evaluate once on
    dls.valid, returning (metrics_row, y_true, y_pred). Same as
    retrain_and_evaluate_hr; used for both final retrains (tuned winner
    and TSSEQUENCER_DEFAULT_PARAMS). depth is not passed.
    """
    set_seed(RANDOM_STATE, reproducible=True)

    metrics = [
        ts.accuracy,
        ts.BalancedAccuracy(),
        ts.F1Score(average="weighted"),
        ts.Precision(average="weighted"),
        ts.Recall(average="weighted"),
        ts.F1Score(average="macro"),
        ts.Precision(average="macro"),
        ts.Recall(average="macro"),
    ]
    loss_func = (
        ts.CrossEntropyLossFlat(weight=dls.train.cws) if use_class_weight else None
    )

    model = ts.create_model(
        ts.TSSequencer,
        dls=dls,
        d_model=params["d_model"],
        fc_dropout=params["fc_dropout"],
    )
    learn = ts.Learner(dls, model, metrics=metrics, loss_func=loss_func)

    t0 = time.time()
    lr_max = learn.lr_find()
    learn.fit_one_cycle(50, lr_max)
    elapsed = time.time() - t0

    vals = learn.recorder.values[-1]

    y_proba, y_true, y_pred = learn.get_preds(with_decoded=True)
    y_proba = y_proba.cpu().numpy()
    y_true = y_true.cpu().numpy()
    y_pred = y_pred.cpu().numpy()

    try:
        roc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted")
    except Exception:
        roc = np.nan

    metrics_row = {
        "d_model": params["d_model"],
        "fc_dropout": params["fc_dropout"],
        "train_loss": vals[0],
        "valid_loss": vals[1],
        "accuracy": vals[2],
        "roc_auc": roc,
        "balanced_acc": vals[3],
        "f1_score": vals[4],
        "precision": vals[5],
        "recall": vals[6],
        "f1_macro": vals[7],
        "precision_macro": vals[8],
        "recall_macro": vals[9],
        "time": elapsed,
    }
    return metrics_row, y_true, y_pred


def tune_and_evaluate_tssequencer_fold(
    X,
    merged,
    col,
    non_test_folds,
    val_folds_list,
    test_fold,
    burst,
    correction_type,
    n_trials=N_TRIALS,
    use_class_weight=True,
):
    """Run one outer test fold for TSSequencer. Same protocol as
    tune_and_evaluate_hr_fold (see its docstring), using
    make_tssequencer_objective, retrain_and_evaluate_tssequencer and
    TSSEQUENCER_DEFAULT_PARAMS.

    Returns:
        tuple: (metrics_df, confusion_df, default_metrics_df,
            default_confusion_df, study). Same shape as
            tune_and_evaluate_hr_fold's return, with model="TSSequencer" /
            "TSSequencer_default".
    """
    study_name = f"{burst.lower()}_{correction_type}_testfold{test_fold}"
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
    study = optuna.create_study(
        direction="maximize", sampler=sampler, study_name=study_name
    )

    study.enqueue_trial(TSSEQUENCER_DEFAULT_PARAMS)

    search_start = time.time()
    study.optimize(
        make_tssequencer_objective(
            X,
            merged,
            col,
            non_test_folds,
            val_folds_list,
            use_class_weight=use_class_weight,
        ),
        n_trials=n_trials,
    )
    search_time = time.time() - search_start
    best_params = study.best_params

    default_trial = next(
        (t for t in study.trials if t.params == TSSEQUENCER_DEFAULT_PARAMS), None
    )
    default_val_f1_macro = default_trial.value if default_trial is not None else np.nan

    try:
        importances = optuna.importance.get_param_importances(study)
    except Exception as e:
        print(f"  Could not compute parameter importances: {e!s}")
        importances = {}

    results = []
    confusion_data = []
    default_results = []
    default_confusion_data = []
    model_name = "TSSequencer"

    try:
        dset = create_dsets_for_split(X, merged, col, non_test_folds, [test_fold])
        dls = ts.TSDataLoaders.from_dsets(dset.train, dset.valid, bs=[64, 128])
        classes = dset.vocab
        n_classes = len(classes)

        tuned_row, y_true, y_pred = retrain_and_evaluate_tssequencer(
            dls, best_params, use_class_weight=use_class_weight
        )
        tuned_row.update(
            {
                "model": model_name,
                "val_f1_macro": study.best_value,
                "default_val_f1_macro": default_val_f1_macro,
                "importance_d_model": importances.get("d_model", np.nan),
                "importance_fc_dropout": importances.get("fc_dropout", np.nan),
                "search_time": search_time,
            }
        )
        results.append(tuned_row)

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
        for actual_idx, actual in enumerate(classes):
            for pred_idx, predicted in enumerate(classes):
                confusion_data.append(
                    {
                        "model": model_name,
                        "actual_label": actual,
                        "predicted_label": predicted,
                        "count": cm[actual_idx, pred_idx],
                    }
                )

        default_row, default_y_true, default_y_pred = retrain_and_evaluate_tssequencer(
            dls, TSSEQUENCER_DEFAULT_PARAMS, use_class_weight=use_class_weight
        )
        default_row["model"] = f"{model_name}_default"
        default_results.append(default_row)

        default_cm = confusion_matrix(
            default_y_true, default_y_pred, labels=np.arange(n_classes)
        )
        for actual_idx, actual in enumerate(classes):
            for pred_idx, predicted in enumerate(classes):
                default_confusion_data.append(
                    {
                        "model": f"{model_name}_default",
                        "actual_label": actual,
                        "predicted_label": predicted,
                        "count": default_cm[actual_idx, pred_idx],
                    }
                )
    except Exception as e:
        print(f"Error with {model_name}: {e!s}")

    results_df = pd.DataFrame(results)
    confusion_df = pd.DataFrame(confusion_data)
    default_results_df = pd.DataFrame(default_results)
    default_confusion_df = pd.DataFrame(default_confusion_data)

    return results_df, confusion_df, default_results_df, default_confusion_df, study

## TSSequencer tuning orchestration, across the 5 outer folds

In [ ]:
def process_parquet_file_tssequencer(
    file_path, fold_lookup_all, n_trials=N_TRIALS, use_class_weight=True
):
    """Run the nested-CV protocol (see notebook intro) for TSSequencer
    hyperparameter tuning + final evaluation, for the behaviour target on
    one file.

    Same as process_parquet_file_hr except for the per-fold trainer
    (tune_and_evaluate_tssequencer_fold) and hyperparameter columns
    (d_model, fc_dropout).
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    acc = pd.read_parquet(file_path)
    X, merged = build_ts_arrays(acc, fold_lookup_all, burst)
    folds = sorted(merged["fold"].unique())
    n_folds = len(folds)

    out = {
        "multiclass_metrics": [],
        "multiclass_conf": [],
        "default_metrics": [],
        "default_conf": [],
        "studies": {},
    }
    col, tgt = "behaviour", "Behaviour"
    for i, test_fold in enumerate(folds):
        non_test_folds = [f for f in folds if f != test_fold]
        val_folds_list = [folds[(i + k) % n_folds] for k in range(1, N_INNER_FOLDS + 1)]
        n_test = int((merged["fold"] == test_fold).sum())

        print(
            f"  Tuning {tgt} classification, test_fold={test_fold}, "
            f"inner val folds={val_folds_list}, non_test_folds={non_test_folds}"
        )
        met, conf, default_met, default_conf, study = (
            tune_and_evaluate_tssequencer_fold(
                X,
                merged,
                col,
                non_test_folds,
                val_folds_list,
                test_fold,
                burst,
                correction_type,
                n_trials=n_trials,
                use_class_weight=use_class_weight,
            )
        )
        out["studies"][test_fold] = study
        if met.empty:
            continue
        for d in (met, conf, default_met, default_conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = tgt
            d["test_fold"] = test_fold
            d["val_folds"] = ",".join(str(vf) for vf in val_folds_list)
            d["n_test_samples"] = n_test
        out["multiclass_metrics"].append(met)
        out["multiclass_conf"].append(conf)
        out["default_metrics"].append(default_met)
        out["default_conf"].append(default_conf)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "d_model",
        "fc_dropout",
        "val_f1_macro",
        "default_val_f1_macro",
        "importance_d_model",
        "importance_fc_dropout",
        "train_loss",
        "valid_loss",
        "accuracy",
        "roc_auc",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "search_time",
        "time",
    ]
    default_metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "d_model",
        "fc_dropout",
        "train_loss",
        "valid_loss",
        "accuracy",
        "roc_auc",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: (
        pd.concat([d for d in lst if not d.empty])[cols]
        if any(not d.empty for d in lst)
        else pd.DataFrame(columns=cols)
    )
    return {
        "multiclass_metrics": fin(out["multiclass_metrics"], metric_columns),
        "multiclass_conf": fin(out["multiclass_conf"], conf_columns),
        "default_metrics": fin(out["default_metrics"], default_metric_columns),
        "default_conf": fin(out["default_conf"], conf_columns),
        "studies": out["studies"],
    }

## Run TSSequencer tuning across datasets

In [ ]:
all_results_tssequencer = []
for burst, correction_type in DATASETS:
    pq_file_path = (
        f"../data/temp/acc_tsai/{burst.capitalize()}/"
        f"annotated_acc_tsai_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file_tssequencer(
        pq_file_path, fold_lookup_all, n_trials=N_TRIALS
    )
    all_results_tssequencer.append(result)

multiclass_metrics_tssequencer = pd.concat(
    [r["multiclass_metrics"] for r in all_results_tssequencer], ignore_index=True
)
multiclass_conf_tssequencer = pd.concat(
    [r["multiclass_conf"] for r in all_results_tssequencer], ignore_index=True
)
default_metrics_tssequencer = pd.concat(
    [r["default_metrics"] for r in all_results_tssequencer], ignore_index=True
)
default_conf_tssequencer = pd.concat(
    [r["default_conf"] for r in all_results_tssequencer], ignore_index=True
)

# Optuna studies kept in memory (not saved) for the "Inspect best study
# per dataset (TSSequencer)" cell.
all_studies_tssequencer = {
    dataset_key: result["studies"]
    for dataset_key, result in zip(DATASETS, all_results_tssequencer)
}

## Pool results and inspect the winning hyperparameters per fold

In [ ]:
def pooled_metrics_from_confusion(conf_df, model_name):
    """Reconstruct pooled y_true/y_pred from a summed confusion matrix and
    compute overall metrics on it -- more meaningful than averaging
    per-fold metrics when classes are unevenly represented across folds.
    """
    sub = conf_df[conf_df["model"] == model_name]
    if sub.empty or sub["count"].sum() == 0:
        return {"model": model_name, "n_pooled_test_samples": 0}
    y_true = np.repeat(sub["actual_label"].to_numpy(), sub["count"].to_numpy())
    y_pred = np.repeat(sub["predicted_label"].to_numpy(), sub["count"].to_numpy())


    return {
        "model": model_name,
        "n_pooled_test_samples": len(y_true),
        "accuracy": float((y_true == y_pred).mean()),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def pooled_summary_per_dataset(conf_df, model_name):
    """Pooled metrics computed separately per (burst, correction_type)."""
    rows = []
    for (burst, correction_type), grp in conf_df.groupby(["burst", "correction_type"]):
        m = pooled_metrics_from_confusion(grp, model_name)
        m["burst"] = burst
        m["correction_type"] = correction_type
        rows.append(m)
    cols = ["burst", "correction_type"] + [
        c for c in rows[0] if c not in ("burst", "correction_type")
    ]
    return pd.DataFrame(rows)[cols]


def _run_df(name):
    """Return the in-memory DataFrame `name` if that model's cells were
    (re-)run this session, else an empty DataFrame.

    Each model's cells can be run on their own, so not every model's
    frames necessarily exist in this session. Models not run are skipped,
    and their rows already on disk are left untouched by upsert_csv.
    """
    return globals().get(name, pd.DataFrame())


# Combine the results of whichever models were run this session into
# single frames, before merging with the files on disk in the save cell.
multiclass_metrics_run = pd.concat(
    [
        df
        for df in (
            _run_df("multiclass_metrics_hr"),
            _run_df("multiclass_metrics_lstm"),
            _run_df("multiclass_metrics_tssequencer"),
        )
        if not df.empty
    ],
    ignore_index=True,
)
multiclass_conf_run = pd.concat(
    [
        df
        for df in (
            _run_df("multiclass_conf_hr"),
            _run_df("multiclass_conf_lstm"),
            _run_df("multiclass_conf_tssequencer"),
        )
        if not df.empty
    ],
    ignore_index=True,
)
default_metrics_run = pd.concat(
    [
        df
        for df in (
            _run_df("default_metrics_hr"),
            _run_df("default_metrics_lstm"),
            _run_df("default_metrics_tssequencer"),
        )
        if not df.empty
    ],
    ignore_index=True,
)
default_conf_run = pd.concat(
    [
        df
        for df in (
            _run_df("default_conf_hr"),
            _run_df("default_conf_lstm"),
            _run_df("default_conf_tssequencer"),
        )
        if not df.empty
    ],
    ignore_index=True,
)

# Only the models present in this run (see _run_df).
MODEL_NAMES = sorted(multiclass_conf_run["model"].unique())
print(f"Models pooled this run: {MODEL_NAMES}")

pooled_multiclass_summary = pd.concat(
    [pooled_summary_per_dataset(multiclass_conf_run, m) for m in MODEL_NAMES],
    ignore_index=True,
)
print("\nPooled tuned multiclass metrics per dataset:")
print(pooled_multiclass_summary)

# Per-fold test results. val_f1_macro vs default_val_f1_macro compares the
# winner with the default on the inner validation folds. Model-specific
# hyperparameter/importance columns are omitted here but kept in
# multiclass_metrics_run and the saved CSV.
per_fold_cols = [
    "burst",
    "correction_type",
    "test_fold",
    "val_folds",
    "n_test_samples",
    "model",
    "val_f1_macro",
    "default_val_f1_macro",
    "accuracy",
    "balanced_acc",
    "f1_macro",
    "search_time",
    "time",
]
per_fold_multiclass = multiclass_metrics_run[per_fold_cols].sort_values(
    ["burst", "correction_type", "model", "test_fold"]
)
print(
    "\nPer-fold tuned multiclass results (final test metrics; see "
    "multiclass_metrics_run for the full per-model hyperparameter/importance columns):"
)
print(per_fold_multiclass)

# Number of outer folds where the winner beat the default on validation,
# and the mean validation gain.
beats_default = (
    per_fold_multiclass["val_f1_macro"] > per_fold_multiclass["default_val_f1_macro"]
)
print(
    f"\nSearch beat the default on validation in {beats_default.sum()}/{len(beats_default)} "
    f"outer folds across {MODEL_NAMES} (mean val_f1_macro - default_val_f1_macro = "
    f"{(per_fold_multiclass['val_f1_macro'] - per_fold_multiclass['default_val_f1_macro']).mean():.4f})"
)

# Test-fold default control: pooled tuned vs pooled default metrics, both
# retrained and evaluated on the same splits with the same seed.
DEFAULT_MODEL_NAMES = sorted(default_conf_run["model"].unique())
pooled_default_summary = pd.concat(
    [pooled_summary_per_dataset(default_conf_run, m) for m in DEFAULT_MODEL_NAMES],
    ignore_index=True,
)
print("\nPooled DEFAULT (test-fold) metrics per dataset:")
print(pooled_default_summary)

# Pair each tuned model with its own "<model>_default" row (merging on
# burst/correction_type alone would cross-join models).
test_gain = pooled_multiclass_summary.assign(
    model_base=pooled_multiclass_summary["model"]
).merge(
    pooled_default_summary.assign(
        model_base=pooled_default_summary["model"].str.replace(
            "_default", "", regex=False
        )
    ),
    on=["burst", "correction_type", "model_base"],
    suffixes=("_tuned", "_default"),
)[["burst", "correction_type", "model_base", "f1_macro_tuned", "f1_macro_default"]]
test_gain["f1_macro_gain"] = test_gain["f1_macro_tuned"] - test_gain["f1_macro_default"]
print("\nTest-set gain (tuned - default, pooled f1_macro), per model:")
print(test_gain)

## Inspect best study per dataset (HydraMultiROCKET)

Post-hoc, after all outer folds are complete. For a chosen dataset, picks
the outer fold with the highest validation `f1_macro` and shows that
fold's Optuna study summary and plots (optimization history, parameter
importances, slice, parallel coordinate).

Change `INSPECT_DATASET_HR` to any tuple from `DATASETS` to look at a
different dataset.

In [ ]:
# Change this to inspect a different dataset -- must be one of the tuples
# in DATASETS.
INSPECT_DATASET_HR = DATASETS[0]

dataset_rows_hr = multiclass_metrics_hr[
    (multiclass_metrics_hr["burst"].str.lower() == INSPECT_DATASET_HR[0])
    & (multiclass_metrics_hr["correction_type"] == INSPECT_DATASET_HR[1])
]
best_row_hr = dataset_rows_hr.sort_values("val_f1_macro", ascending=False).iloc[0]
best_test_fold_hr = best_row_hr["test_fold"]
study_hr = all_studies_hr[INSPECT_DATASET_HR][best_test_fold_hr]

complete_trials_hr = [
    t for t in study_hr.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(
    f"Best-performing outer fold for {INSPECT_DATASET_HR} "
    f"(by validation f1_macro across its {len(dataset_rows_hr)} outer folds): "
    f"test_fold={best_test_fold_hr}"
)
print(f"  # trials (completed) : {len(complete_trials_hr)}/{len(study_hr.trials)}")
print(f"  Best value            : {study_hr.best_value:.4f}")
print(f"  Best params           : {study_hr.best_params}")
print(f"  Default params        : {HR_DEFAULT_PARAMS}")
print(f"  Default value         : {best_row_hr['default_val_f1_macro']:.4f}")
print(
    f"  Gain vs default       : {study_hr.best_value - best_row_hr['default_val_f1_macro']:+.4f}"
)

for plot_fn in [
    optuna.visualization.plot_optimization_history,
    optuna.visualization.plot_param_importances,
    optuna.visualization.plot_slice,
    optuna.visualization.plot_parallel_coordinate,
]:
    try:
        plot_fn(study_hr).show()
    except Exception as e:
        print(f"  Could not generate {plot_fn.__name__}: {e!s}")

## Inspect best study per dataset (LSTM)

Same as the HydraMultiROCKET cell above.

In [ ]:
# Change this to inspect a different dataset -- must be one of the tuples
# in DATASETS.
INSPECT_DATASET_LSTM = DATASETS[0]

dataset_rows_lstm = multiclass_metrics_lstm[
    (multiclass_metrics_lstm["burst"].str.lower() == INSPECT_DATASET_LSTM[0])
    & (multiclass_metrics_lstm["correction_type"] == INSPECT_DATASET_LSTM[1])
]
best_row_lstm = dataset_rows_lstm.sort_values("val_f1_macro", ascending=False).iloc[0]
best_test_fold_lstm = best_row_lstm["test_fold"]
study_lstm = all_studies_lstm[INSPECT_DATASET_LSTM][best_test_fold_lstm]

complete_trials_lstm = [
    t for t in study_lstm.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(
    f"Best-performing outer fold for {INSPECT_DATASET_LSTM} "
    f"(by validation f1_macro across its {len(dataset_rows_lstm)} outer folds): "
    f"test_fold={best_test_fold_lstm}"
)
print(f"  # trials (completed) : {len(complete_trials_lstm)}/{len(study_lstm.trials)}")
print(f"  Best value            : {study_lstm.best_value:.4f}")
print(f"  Best params           : {study_lstm.best_params}")
print(f"  Default params        : {LSTM_DEFAULT_PARAMS}")
print(f"  Default value         : {best_row_lstm['default_val_f1_macro']:.4f}")
print(
    f"  Gain vs default       : {study_lstm.best_value - best_row_lstm['default_val_f1_macro']:+.4f}"
)

for plot_fn in [
    optuna.visualization.plot_optimization_history,
    optuna.visualization.plot_param_importances,
    optuna.visualization.plot_slice,
    optuna.visualization.plot_parallel_coordinate,
]:
    try:
        plot_fn(study_lstm).show()
    except Exception as e:
        print(f"  Could not generate {plot_fn.__name__}: {e!s}")

## Inspect best study per dataset (TSSequencer)

Same as the HydraMultiROCKET cell above.

In [ ]:
# Change this to inspect a different dataset -- must be one of the tuples
# in DATASETS.
INSPECT_DATASET_TSS = DATASETS[0]

dataset_rows_tss = multiclass_metrics_tssequencer[
    (multiclass_metrics_tssequencer["burst"].str.lower() == INSPECT_DATASET_TSS[0])
    & (multiclass_metrics_tssequencer["correction_type"] == INSPECT_DATASET_TSS[1])
]
best_row_tss = dataset_rows_tss.sort_values("val_f1_macro", ascending=False).iloc[0]
best_test_fold_tss = best_row_tss["test_fold"]
study_tss = all_studies_tssequencer[INSPECT_DATASET_TSS][best_test_fold_tss]

complete_trials_tss = [
    t for t in study_tss.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(
    f"Best-performing outer fold for {INSPECT_DATASET_TSS} "
    f"(by validation f1_macro across its {len(dataset_rows_tss)} outer folds): "
    f"test_fold={best_test_fold_tss}"
)
print(f"  # trials (completed) : {len(complete_trials_tss)}/{len(study_tss.trials)}")
print(f"  Best value            : {study_tss.best_value:.4f}")
print(f"  Best params           : {study_tss.best_params}")
print(f"  Default params        : {TSSEQUENCER_DEFAULT_PARAMS}")
print(f"  Default value         : {best_row_tss['default_val_f1_macro']:.4f}")
print(
    f"  Gain vs default       : {study_tss.best_value - best_row_tss['default_val_f1_macro']:+.4f}"
)

for plot_fn in [
    optuna.visualization.plot_optimization_history,
    optuna.visualization.plot_param_importances,
    optuna.visualization.plot_slice,
    optuna.visualization.plot_parallel_coordinate,
]:
    try:
        plot_fn(study_tss).show()
    except Exception as e:
        print(f"  Could not generate {plot_fn.__name__}: {e!s}")

## Helper: merge new results into existing output CSVs

In [ ]:
def upsert_csv(new_df, path, key_cols):
    """Merge new_df's rows into the CSV at path (creating it if it
    doesn't exist yet), keeping new_df's rows wherever key_cols collide
    with what's already on disk.

    Lets each model's cells be run independently: rows for models not in
    new_df are kept as they are on disk.
    """
    if os.path.exists(path):
        existing = pd.read_csv(path)
        combined = pd.concat([existing, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=key_cols, keep="last")
    else:
        combined = new_df
    combined.to_csv(path, index=False)
    return combined

## Save outputs

In [ ]:
# Merges this session's results (built in the pooling cell) into the
# output CSVs via upsert_csv, so rows for models not run this session are
# kept. Only the behaviour target is saved.
METRIC_KEY = ["burst", "correction_type", "target", "test_fold", "model"]
CONF_KEY = [
    "burst",
    "correction_type",
    "target",
    "test_fold",
    "model",
    "actual_label",
    "predicted_label",
]

upsert_csv(
    multiclass_metrics_run,
    "../data/output/comparison_behaviour/behaviour_acceleration_dl_tuned_metrics.csv",
    METRIC_KEY,
)
upsert_csv(
    multiclass_conf_run,
    "../data/output/comparison_behaviour/behaviour_acceleration_dl_tuned_confusion.csv",
    CONF_KEY,
)

# Default-hyperparameter control (test-fold retrain), saved to separate
# files so the tuned metrics/confusion files read by
# 12b_publication_plots_fig3.R contain only tuned rows.
upsert_csv(
    default_metrics_run,
    "../data/output/comparison_behaviour/behaviour_acceleration_dl_tuned_default_control_metrics.csv",
    METRIC_KEY,
)
upsert_csv(
    default_conf_run,
    "../data/output/comparison_behaviour/behaviour_acceleration_dl_tuned_default_control_confusion.csv",
    CONF_KEY,
)